# Results Analysis Notebook

## Introduction

This notebook analyzes model training results, predictions, and performance metrics.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))

%matplotlib inline


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2

def generate_gradcam(model, image, layer_name='top_conv'):
    # Create a model that outputs the last conv layer + predictions
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(layer_name).output,
            model.output
        ]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(image)
        # Target class 1 = fake
        loss = predictions[:, 1]

    # Gradient of fake score w.r.t. last conv feature map
    grads = tape.gradient(loss, conv_outputs)

    # Global average pool the gradients
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weighted sum of feature maps
    cam = tf.reduce_sum(
        tf.multiply(weights, conv_outputs[0]), axis=-1
    )

    # ReLU and normalize
    cam = np.maximum(cam.numpy(), 0)
    cam = cam / cam.max()

    # Resize to original image size
    cam = cv2.resize(cam, (380, 380))

    return cam

def overlay_heatmap(original_image, cam):
    heatmap = cv2.applyColorMap(
        np.uint8(255 * cam), cv2.COLORMAP_JET
    )
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(original_image, 0.6, heatmap, 0.4, 0)
    return overlay

# Load a test image
image = preprocess('test_fake_face.jpg')  # your preprocessing function

# Generate
cam = generate_gradcam(model, image)
overlay = overlay_heatmap(original_image, cam)

# Show side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(original_image)
axes[0].set_title('Original Image')
axes[1].imshow(cam, cmap='jet')
axes[1].set_title('Grad-CAM Heatmap')
axes[2].imshow(overlay)
axes[2].set_title('Overlay — Where Model Looked')
plt.savefig('gradcam_result.png', dpi=150)
plt.show()